# Fish detector - Colab training (FathomNet deep-sea fish + TrashCan)

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`.

**Upload first:** `fish_dataset.zip` (made locally by `python -m src.data.build_fish_dataset` + zip) to
`MyDrive/fish/fish_dataset.zip`.

- train = FathomNet deep-sea fish + TrashCan train fish + 1000 TrashCan "no fish" images
- val   = TrashCan val (fish only) -> picks the best epoch
- test  = TrashCan test -> **not used here**; the final precision/recall is computed locally on CPU with
  the same scoring code as the current model, so the comparison is fair.

**Bring back:** `MyDrive/fish/best_fish.pt` -> `models/fish_detector/best_fish.pt` in the project.
If Colab disconnects: set `RESUME = True` and run all again.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
EPOCHS = 60
PATIENCE = 15
BATCH = 16
IMGSZ = 640
RUN_NAME = 'yolo11s_fish'
RESUME = False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, yaml
DRIVE_DIR = '/content/drive/MyDrive/fish'
ZIP = f'{DRIVE_DIR}/fish_dataset.zip'
assert os.path.exists(ZIP), f'Upload fish_dataset.zip to {ZIP}'
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/fish')
root = '/content/fish/processed_fish'
cfg = yaml.safe_load(open(f'{root}/data.yaml'))
cfg['path'] = root
yaml.safe_dump(cfg, open(f'{root}/data.yaml', 'w'), sort_keys=False)
for s in ('train', 'val'):
    print(s, len(os.listdir(f'{root}/{s}/images')), 'images')

In [ ]:
from ultralytics import YOLO
from pathlib import Path
PROJECT = f'{DRIVE_DIR}/runs'
last = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
if RESUME:
    assert last.exists(), f'No checkpoint at {last}'
    YOLO(str(last)).train(resume=True)
else:
    YOLO('yolo11s.pt').train(
        data=f'{root}/data.yaml', epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ,
        device=0, seed=42, cos_lr=True, cache='ram', project=PROJECT, name=RUN_NAME, exist_ok=True, plots=True,
    )

In [ ]:
import shutil
best = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
shutil.copyfile(best, f'{DRIVE_DIR}/best_fish.pt')
m = YOLO(str(best)).val(data=f'{root}/data.yaml', split='val', imgsz=IMGSZ, device=0)
print(f'VAL (TrashCan fish): precision {m.box.mp:.3f}  recall {m.box.mr:.3f}  mAP50 {m.box.map50:.3f}')
print('Saved', f'{DRIVE_DIR}/best_fish.pt', '-> download it into models/fish_detector/')